# Case study: a data quality report for the BTI decisions

*Session 4, block 1 · Author: course team · Licence: CC-BY-4.0*

**Task.** Write a data quality report for the course data: which checks did you run, what did they find,
how severe is each finding, and what should happen next?

The notebook expresses every check as code, so that the report can be regenerated after each data update.
Sections 2–3 check the prepared training table `train.parquet` (309,529 decisions, 2017–2023) and the
nomenclature; section 4 looks at the **raw export** of the European Commission, where some problems are
still visible that `prepare_data.py` already removed. Section 6 writes the rules as a **pandera** schema, and
the file [`quality/test_bti_quality.py`](quality/test_bti_quality.py) contains them as **pytest** tests.

Cells marked **TODO** are for you.

In [ ]:
from pathlib import Path

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data"
assert (DATA / "train.parquet").exists(), "run case-study/prepare_data.py first"

import numpy as np
import pandas as pd

decisions = pd.read_parquet(DATA / "train.parquet")
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet")
print(decisions.shape, nomenclature.shape)

## 1. A small framework: every check returns a row

Each check records the **dimension** of data quality it belongs to, the table and column, the number of
failing rows and a **severity**:

- *critical*: breaks a key or a hard rule; the data cannot be used as they are;
- *major*: affects many rows or a central variable; must be handled before analysis;
- *minor*: worth documenting; handle when it matters for a question.

In [ ]:
results = []


def check(dimension, table, column, description, failed, n_total, severity):
    """Record one check. `failed` is a boolean Series (True = row fails) or a count."""
    n_failed = int(failed.sum()) if hasattr(failed, "sum") else int(failed)
    results.append({"dimension": dimension, "table": table, "column": column, "check": description,
                    "n_failed": n_failed, "share": n_failed / n_total,
                    "severity": severity if n_failed else "ok"})

## 2. Checks on the decisions

### Completeness: missing values and hidden missing values

A value can be missing as `NULL`/`NaN` or *hidden* as an empty string, a placeholder ("n/a", "-") or a
default value such as the date 1900-01-01. Both count as incomplete. Some values are missing **by
design**: a decision that simply expired, or is still valid, has no invalidation reason. This is called
**structural missingness**; it is not a defect, but it must be documented.

In [ ]:
n = len(decisions)
for col in decisions.columns:
    check("completeness", "decisions", col, "value is NULL", decisions[col].isna(), n,
          "minor" if col in ("keywords", "invalidation_reason") else "critical")
check("completeness", "decisions", "description", "description shorter than 20 characters",
      decisions["description"].str.strip().str.len().lt(20), n, "minor")
check("completeness", "decisions", "description", "description is a database template ('SQL{...}')",
      decisions["description"].str.upper().str.contains("SQL{", regex=False), n, "major")
check("completeness", "decisions", "end_date", "placeholder date 1900-01-01 (hidden missing)",
      decisions["end_date"].dt.year.eq(1900), n, "major")

In [ ]:
# structural missingness: who has an invalidation reason?
pd.crosstab(decisions["status"], decisions["invalidation_reason"].notna().rename("has_reason"), margins=True)

Every valid decision has no reason, as expected. Of the invalid decisions, 45,223 were invalidated before
their normal end; the other 254,884 simply expired after three years. The codes are documented in the
Commission's *Administrative guidance on the BTI process*; the most frequent here are 61 (nomenclature code
changed), 98 (Brexit), 64 (incorrect classification), 68 (EORI number of the holder changed), 65 (other
reasons than classification) and 55 (annulled).

In [ ]:
decisions["invalidation_reason"].value_counts(dropna=False)

### Validity: types, formats and ranges

In [ ]:
expected_types = {"start_date": "datetime64[ns]", "end_date": "datetime64[ns]", "date_of_issue": "datetime64[ns]",
                  "heading": "object", "cn_code": "object", "description": "object"}
for col, dtype in expected_types.items():
    check("validity", "decisions", col, f"dtype is {dtype}", int(str(decisions[col].dtype) != dtype), 1, "critical")

check("validity", "decisions", "heading", "heading has four digits",
      ~decisions["heading"].str.fullmatch(r"\d{4}"), n, "critical")
check("validity", "decisions", "cn_code", "CN code has eight digits",
      ~decisions["cn_code"].str.fullmatch(r"\d{8}"), n, "minor")
check("validity", "decisions", "issuing_country", "two upper-case letters",
      ~decisions["issuing_country"].str.fullmatch(r"[A-Z]{2}"), n, "critical")
check("validity", "decisions", "language", "two lower-case letters",
      ~decisions["language"].str.fullmatch(r"[a-z]{2}"), n, "critical")
check("validity", "decisions", "status", "status in {VALID, INVALID}",
      ~decisions["status"].isin(["VALID", "INVALID"]), n, "critical")
check("validity", "decisions", "start_date", "start date in 2017-2023 (training period)",
      ~decisions["start_date"].between("2017-01-01", "2023-12-31"), n, "critical")
check("validity", "decisions", "end_date", "end date not before start date",
      decisions["end_date"].lt(decisions["start_date"]), n, "major")
check("validity", "decisions", "date_of_issue", "issued not after the start of validity",
      decisions["date_of_issue"].gt(decisions["start_date"]), n, "major")

In [ ]:
# which decisions end before they start?
early_end = decisions["end_date"].lt(decisions["start_date"])
print(decisions.loc[early_end, "end_date"].value_counts().head(3))
print(decisions.loc[early_end, "invalidation_reason"].value_counts())

All 510 decisions that "end before they start" carry the end date 1900-01-01 and the invalidation code 55
(annulled). The date is a **placeholder** for "annulled from the beginning", not a typing error. The right
treatment is therefore to set it to missing and keep the code, not to correct a digit.

### Uniqueness: keys and duplicates

In [ ]:
check("uniqueness", "decisions", "bti_reference", "bti_reference unique",
      decisions["bti_reference"].duplicated(), n, "critical")
check("uniqueness", "decisions", "description", "description appears more than once (exact)",
      decisions["description"].duplicated(), n, "major")
normalised = decisions["description"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
check("uniqueness", "decisions", "description", "description appears more than once (ignoring case and spaces)",
      normalised.duplicated(), n, "major")
decisions["description"].str.slice(0, 60).value_counts().head(5)

In [ ]:
# are repeated descriptions classified consistently?
groups = decisions.assign(norm=normalised)[normalised.duplicated(keep=False)].groupby("norm")
per_text = groups.agg(n=("heading", "size"), n_headings=("heading", "nunique"),
                      n_countries=("issuing_country", "nunique"), first=("start_date", "min"), last=("start_date", "max"))
print(len(per_text), "repeated descriptions;", (per_text["n_headings"] > 1).sum(), "of them under more than one heading")
per_text.sort_values("n", ascending=False).head(5)

The most repeated "description" is not a description at all: 63 decisions contain the text of a database
template (`sql{decode(nvl([neu],0), ...`) instead of the goods description, and they are classified under 34
different headings. With small variants, 73 German decisions of 2017–2020 are affected (check below). Such rows carry no information about the product and should be removed before training
a classifier.

Most other repeated descriptions are **renewals** (the same product, a new decision after three years) or
series of similar products filed together. For a classifier, they matter twice: a random train/test split puts
copies on both sides and makes the score optimistic (Session 7), and a description that was classified under
two headings is either a classification change or an error.

### Consistency: rules that relate columns and tables

In [ ]:
check("consistency", "decisions", "chapter", "chapter = first two digits of the heading",
      decisions["chapter"].ne(decisions["heading"].str[:2]), n, "critical")
check("consistency", "decisions", "cn_code", "CN code starts with the heading",
      decisions["cn_code"].str[:4].ne(decisions["heading"]), n, "critical")
check("consistency", "decisions", "heading", "heading exists in the HS 2022 nomenclature (foreign key)",
      ~decisions["heading"].isin(nomenclature["heading"]), n, "major")
check("consistency", "decisions", "invalidation_reason", "a valid decision has no invalidation reason",
      decisions["status"].eq("VALID") & decisions["invalidation_reason"].notna(), n, "critical")
validity_days = (decisions["end_date"] - decisions["start_date"]).dt.days
check("consistency", "decisions", "end_date", "no reason, but validity shorter than three years",
      decisions["invalidation_reason"].isna() & validity_days.lt(1094), n, "minor")
check("consistency", "decisions", "end_date", "invalidation reason, but full three-year validity",
      decisions["invalidation_reason"].notna() & validity_days.ge(1094), n, "minor")

In [ ]:
# language and issuing country: which languages does each country use?
OFFICIAL = {"AT": {"de"}, "BE": {"nl", "fr", "de"}, "BG": {"bg"}, "CY": {"el"}, "CZ": {"cs"}, "DE": {"de"},
            "DK": {"da"}, "EE": {"et"}, "ES": {"es"}, "FI": {"fi", "sv"}, "FR": {"fr"}, "GB": {"en"},
            "GR": {"el"}, "HR": {"hr"}, "HU": {"hu"}, "IE": {"en", "ga"}, "IT": {"it"}, "LT": {"lt"},
            "LU": {"fr", "de"}, "LV": {"lv"}, "MT": {"mt", "en"}, "NL": {"nl"}, "PL": {"pl"}, "PT": {"pt"},
            "RO": {"ro"}, "SE": {"sv"}, "SI": {"sl"}, "SK": {"sk"}, "XI": {"en"}}
assert set(decisions["issuing_country"]) <= set(OFFICIAL), "a country is missing in OFFICIAL"
official = [lang in OFFICIAL[c] for c, lang in zip(decisions["issuing_country"], decisions["language"])]
other = ~pd.Series(official, index=decisions.index)
check("consistency", "decisions", "language", "language is an official language of the issuing country",
      other & decisions["language"].ne("en"), n, "minor")
check("consistency", "decisions", "language", "English, but not an official language of the country",
      other & decisions["language"].eq("en"), n, "minor")
decisions.loc[other & decisions["language"].ne("en")].groupby(["issuing_country", "language"]).size()

English descriptions from non-English-speaking countries are plausible (traders may file in English).
A Dutch decision written in Greek or Maltese, or an Italian one in Lithuanian, is more likely a wrong
language code: a minor finding, but one that matters for any language-specific text processing (Session 13).

## 3. Checks on the nomenclature

In [ ]:
m = len(nomenclature)
check("uniqueness", "nomenclature", "heading", "heading unique", nomenclature["heading"].duplicated(), m, "critical")
check("validity", "nomenclature", "heading", "heading has four digits",
      ~nomenclature["heading"].str.fullmatch(r"\d{4}"), m, "critical")
check("consistency", "nomenclature", "chapter", "chapter = first two digits of the heading",
      nomenclature["chapter"].ne(nomenclature["heading"].str[:2]), m, "critical")
for col in nomenclature.columns:
    check("completeness", "nomenclature", col, "value is NULL", nomenclature[col].isna(), m, "major")
check("completeness", "nomenclature", "heading", "heading without any training decision",
      ~nomenclature["heading"].isin(decisions["heading"]), m, "minor")

## 4. A look at the raw export

`prepare_data.py` already parsed the dates and removed decisions outside 2017–2026. The raw export shows
what it had to deal with. We read four columns of all 23 yearly files directly from the zip archive (this
takes about half a minute).

In [ ]:
import zipfile

ZIP = DATA / "raw" / "DDS2-EBTI_Full.zip"
cols = ["BTI_REFERENCE", "START_DATE_OF_VALIDITY", "END_DATE_OF_VALIDITY", "NOMENCLATURE_CODE"]
with zipfile.ZipFile(ZIP) as z:
    raw = pd.concat([pd.read_csv(z.open(name), usecols=cols, dtype=str, encoding="utf-8-sig")
                     for name in sorted(z.namelist())], ignore_index=True)
print(f"{len(raw):,} decisions in the raw export")
raw.head(3)

In [ ]:
# dates are text in the format dd/mm/yyyy; parse them explicitly and count failures
start = pd.to_datetime(raw["START_DATE_OF_VALIDITY"], format="%d/%m/%Y", errors="coerce")
end = pd.to_datetime(raw["END_DATE_OF_VALIDITY"], format="%d/%m/%Y", errors="coerce")
print("start dates that do not parse:", start.isna().sum(), "| end dates that do not parse:", end.isna().sum())
print("start dates after 2026:")
print(raw.loc[start.dt.year > 2026, ["BTI_REFERENCE", "START_DATE_OF_VALIDITY", "END_DATE_OF_VALIDITY"]])

Three start dates lie in 2027, 2055 and 2200: typing errors that no plausibility rule at the source caught.
Without `format=`, pandas would have to guess whether `05/06/2023` is 5 June or 6 May; with it, the rule is
explicit and every failure becomes a count.

In [ ]:
# the nomenclature code: ten digits of the TARIC code, then asterisks and optional additional codes
pattern = raw["NOMENCLATURE_CODE"].str.replace(r"\d", "9", regex=True)
print(pattern.value_counts().head(6))
digits = raw["NOMENCLATURE_CODE"].str.replace(r"\D", "", regex=True)
print("codes with fewer than 4 digits:", digits.str.len().lt(4).sum())

**TODO 1.** How many raw decisions have an end date before their start date, and how many of them use the
placeholder 1900-01-01? Is the share the same in every year?

In [ ]:
# TODO 1

## 5. The report

In [ ]:
order = {"critical": 0, "major": 1, "minor": 2, "ok": 3}
report = (pd.DataFrame(results)
          .assign(rank=lambda d: d["severity"].map(order), share=lambda d: d["share"].round(4))
          .sort_values(["rank", "n_failed"], ascending=[True, False])
          .drop(columns="rank")
          .reset_index(drop=True))
pd.set_option("display.max_colwidth", 70)
report[report["severity"] != "ok"]

In [ ]:
print(f"{len(report)} checks, {(report['severity'] == 'ok').sum()} passed")
report.groupby(["dimension", "severity"]).size().unstack(fill_value=0)

In [ ]:
def to_markdown(df: pd.DataFrame) -> str:
    """A small Markdown table writer (avoids an extra dependency)."""
    lines = ["| " + " | ".join(df.columns) + " |", "|" + "---|" * len(df.columns)]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)


findings = report[report["severity"] != "ok"][["severity", "dimension", "table", "column", "check", "n_failed", "share"]]
text = f"""# Data quality report: BTI decisions

Data: `train.parquet` ({len(decisions):,} decisions) and `nomenclature.parquet` ({len(nomenclature):,} headings).
Checks run: {len(report)}; passed: {(report['severity'] == 'ok').sum()}.

## Findings

{to_markdown(findings)}

## Recommended actions

1. (TODO) ...
"""
print(text[:3000])

**TODO 2.** Write the *Recommended actions*: for each major finding, say what should be done (remove,
correct, flag, impute, document) and why. Example: "end_date 1900-01-01 for 510 annulled decisions: set to
missing, keep the code 55, document; do not compute a validity duration for them."

**TODO 3.** Add two checks of your own, for example: does `keywords` contain duplicates within one decision?
Are there decisions whose description is identical to the description of a test decision?

In [ ]:
# TODO 3

## 6. The same rules as a pandera schema

[pandera](https://pandera.readthedocs.io/) (MIT licence) describes the expected structure of a DataFrame as a
**schema**: columns, types and checks. `validate(lazy=True)` runs all checks and collects every failure
instead of stopping at the first one. Workbook 01 introduces pandera in general. pandera is part of the course
environment; otherwise start Jupyter with `uv run --with pandera jupyter lab`.

In [ ]:
try:
    import pandera.pandas as pa
except ImportError:
    pa = None
    print("pandera is not installed: start Jupyter with  uv run --with pandera jupyter lab")

if pa is not None:
    decision_schema = pa.DataFrameSchema(
        {
            "bti_reference": pa.Column(str, unique=True),
            "issuing_country": pa.Column(str, pa.Check.str_matches(r"^[A-Z]{2}$")),
            "language": pa.Column(str, pa.Check.str_matches(r"^[a-z]{2}$")),
            "start_date": pa.Column("datetime64[ns]", pa.Check.in_range(pd.Timestamp("2017-01-01"), pd.Timestamp("2023-12-31"))),
            "status": pa.Column(str, pa.Check.isin(["VALID", "INVALID"])),
            "description": pa.Column(str, pa.Check(lambda s: s.str.strip().str.len() > 0, error="description is empty")),
            "heading": pa.Column(str, [pa.Check.str_matches(r"^\d{4}$"),
                                       pa.Check.isin(nomenclature["heading"], error="heading not in HS 2022")]),
            "cn_code": pa.Column(str, pa.Check.str_matches(r"^\d{8}$")),
            "end_date": pa.Column("datetime64[ns]", pa.Check.ge(pd.Timestamp("2017-01-01"),
                                                                 error="end date before 2017 (placeholder?)")),
        },
    )
    try:
        decision_schema.validate(decisions, lazy=True)
    except pa.errors.SchemaErrors as err:
        print(err.failure_cases.groupby(["column", "check"], dropna=False).size())

## 7. Validation rules as tests

A **test** is a function that fails loudly when a rule is broken. Running the tests after every data update
(locally or in continuous integration, Session 2) turns the quality report into a guard. From the repository
root:

```bash
uv run --with pytest pytest sessions/04-data-quality/workbooks/quality -v
```

Known problems of the data are marked as *expected failures* (`xfail`) with the reason, so that they are
documented without making the test run fail. After block 3, the tests on the cleaned table must pass.

In [ ]:
test_file = Path("quality") / "test_bti_quality.py"
if not test_file.exists():
    test_file = ROOT / "sessions/04-data-quality/workbooks/quality/test_bti_quality.py"
print(test_file.read_text()[:2500])